<a href="https://colab.research.google.com/github/fabianxox/machinelearning/blob/main/naive_bayes_implementation_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd

data = {
    "OFFER":   ["Yes", "Yes", "No",  "No",  "Yes", "No",  "Yes", "No"],
    "URGENT":  ["Yes", "No",  "Yes", "No",  "No",  "Yes", "Yes", "No"],
    "LINK":    ["Yes", "Yes", "No",  "No",  "Yes", "No",  "No",  "Yes"],
    "Spam":    ["Yes", "Yes", "No",  "No",  "Yes", "No",  "Yes", "No"]
}

In [38]:
def load_dataset(data):
  table= pd.DataFrame(data)
  return table

In [39]:
data= load_dataset(data)
print(data)

  OFFER URGENT LINK Spam
0   Yes    Yes  Yes  Yes
1   Yes     No  Yes  Yes
2    No    Yes   No   No
3    No     No   No   No
4   Yes     No  Yes  Yes
5    No    Yes   No   No
6   Yes    Yes   No  Yes
7    No     No  Yes   No


In [40]:
def get_counts(data):
  prior_counts= {}
  counts= {}
  for row in range(data.shape[0]):
    class_name= data["Spam"][row]
    if class_name not in prior_counts:
      prior_counts[class_name]= 1
    else:
      prior_counts[class_name]+=1
    for feature in data.columns:
      if feature== "Spam":
        continue
      value= data[feature][row]
      if feature not in counts:
        counts[feature]= {}
      if class_name not in counts[feature]:
        counts[feature][class_name]={
            "Yes": 0,
            "No": 0
        }
      counts[feature][class_name][value]+=1
  return counts, prior_counts



In [41]:
count, prior_counts= get_counts(data)
print(count)
print(prior_counts)
c= pd.DataFrame(count)
print(c)

{'OFFER': {'Yes': {'Yes': 4, 'No': 0}, 'No': {'Yes': 0, 'No': 4}}, 'URGENT': {'Yes': {'Yes': 2, 'No': 2}, 'No': {'Yes': 2, 'No': 2}}, 'LINK': {'Yes': {'Yes': 3, 'No': 1}, 'No': {'Yes': 1, 'No': 3}}}
{'Yes': 4, 'No': 4}
                   OFFER               URGENT                 LINK
Yes  {'Yes': 4, 'No': 0}  {'Yes': 2, 'No': 2}  {'Yes': 3, 'No': 1}
No   {'Yes': 0, 'No': 4}  {'Yes': 2, 'No': 2}  {'Yes': 1, 'No': 3}


In [42]:
def calculate_probability(count, prior_counts):
  probs= {}
  for feature in count:
    for class_name in count[feature]:
      for key, value in count[feature][class_name].items():
        if feature not in probs:
          probs[feature]= {}
        if class_name not in probs[feature]:
          probs[feature][class_name]= {
                "Yes": 0,
                "No": 0
            }


        probs[feature][class_name][key]= laplace_smooth(value, prior_counts[class_name])


  return probs

In [43]:
def laplace_smooth(value, total):
  return (value+1)/(total+2)

In [44]:
probability= calculate_probability(count, prior_counts)
print(probability)

{'OFFER': {'Yes': {'Yes': 0.8333333333333334, 'No': 0.16666666666666666}, 'No': {'Yes': 0.16666666666666666, 'No': 0.8333333333333334}}, 'URGENT': {'Yes': {'Yes': 0.5, 'No': 0.5}, 'No': {'Yes': 0.5, 'No': 0.5}}, 'LINK': {'Yes': {'Yes': 0.6666666666666666, 'No': 0.3333333333333333}, 'No': {'Yes': 0.3333333333333333, 'No': 0.6666666666666666}}}


In [45]:
def prior_prob(prior_counts):
  total= sum(prior_counts.values())
  prior_probability= {}
  for key, value in prior_counts.items():
    prior_probability[key]= value/total

  return prior_probability

In [46]:
prio_prob=  prior_prob(prior_counts)
print(prio_prob)

print(probability)

{'Yes': 0.5, 'No': 0.5}
{'OFFER': {'Yes': {'Yes': 0.8333333333333334, 'No': 0.16666666666666666}, 'No': {'Yes': 0.16666666666666666, 'No': 0.8333333333333334}}, 'URGENT': {'Yes': {'Yes': 0.5, 'No': 0.5}, 'No': {'Yes': 0.5, 'No': 0.5}}, 'LINK': {'Yes': {'Yes': 0.6666666666666666, 'No': 0.3333333333333333}, 'No': {'Yes': 0.3333333333333333, 'No': 0.6666666666666666}}}


In [47]:
def predict(mail, prior_probability, probability):
  prediction={}
  for word, value in mail.items():
      for class_name in probability[word]:
        if class_name not in prediction:
          prediction[class_name]= prior_probability[class_name]
        prediction[class_name]*= probability[word][class_name][value]
  p= 0
  category="x"

  for key, value in prediction.items():
    print(f"key: {key}, value: {value}")
    if value>p:
      p= value
      category= key

  return category, p

In [48]:
mail = {
    "OFFER": "Yes",
    "URGENT": "Yes",
    "LINK": "No"
}

prediction, proba= predict(mail, prio_prob, probability)
print(prediction, proba)

key: Yes, value: 0.06944444444444445
key: No, value: 0.027777777777777776
Yes 0.06944444444444445


In [51]:
def evaluate(data, prior_prob, probability):
    correct = 0

    for row in range(data.shape[0]):

        email = {}

        for feature in data.columns:
            if feature == "Spam":
                continue
            email[feature] = data[feature][row]

        actual = data["Spam"][row]

        category, score = predict(email, prior_prob, probability)

        if category == actual:
            correct += 1

        print(
            f"Row {row+1}: Predicted = {category} ({score}), Actual = {actual}"
        )

    accuracy = correct / data.shape[0]
    return accuracy

In [55]:
prior = prior_prob(prior_counts)

In [56]:
eval= evaluate(data, prior, probability)
print(eval)

key: Yes, value: 0.1388888888888889
key: No, value: 0.013888888888888888
Row 1: Predicted = Yes (0.1388888888888889), Actual = Yes
key: Yes, value: 0.1388888888888889
key: No, value: 0.013888888888888888
Row 2: Predicted = Yes (0.1388888888888889), Actual = Yes
key: Yes, value: 0.013888888888888888
key: No, value: 0.1388888888888889
Row 3: Predicted = No (0.1388888888888889), Actual = No
key: Yes, value: 0.013888888888888888
key: No, value: 0.1388888888888889
Row 4: Predicted = No (0.1388888888888889), Actual = No
key: Yes, value: 0.1388888888888889
key: No, value: 0.013888888888888888
Row 5: Predicted = Yes (0.1388888888888889), Actual = Yes
key: Yes, value: 0.013888888888888888
key: No, value: 0.1388888888888889
Row 6: Predicted = No (0.1388888888888889), Actual = No
key: Yes, value: 0.06944444444444445
key: No, value: 0.027777777777777776
Row 7: Predicted = Yes (0.06944444444444445), Actual = Yes
key: Yes, value: 0.027777777777777776
key: No, value: 0.06944444444444445
Row 8: Predic